In [1]:
from google import genai
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

In [9]:
complaint = input("Enter your complaint: ")

print("\nCustomer Complaint:")
print(complaint)

Enter your complaint:  Amount not refund



Customer Complaint:
Amount not refund


In [10]:
def classify_complaint(complaint):

    prompt = f"""
    You are an e-commerce customer complaint classification system.

    Analyze the following customer complaint.

    Complaint:
    {complaint}

    Choose exactly one category:

    - Order Issue
    - Delivery Issue
    - Product Issue
    - Return/Replacement
    - Refund Issue
    - Payment Issue
    - Account Issue
    - Cancellation
    - General Inquiry

    Choose exactly one priority:

    - Low
    - Medium
    - High

    Return exactly:

    Category: <category>
    Priority: <priority>
    """

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return response.text

In [11]:
classification_result = classify_complaint(complaint)

print("Classification Result:")
print(classification_result)

Classification Result:
Category: Refund Issue
Priority: High


In [12]:
def process_complaint(complaint):

    prompt = f"""
    You are an e-commerce customer support input processing system.

    Analyze the following customer complaint.

    Customer Complaint:
    {complaint}

    Extract:

    1. Order ID
    2. Product
    3. Issue
    4. Customer Request
    5. Date

    Rules:
    - Extract information only if it is present.
    - Do not invent or assume information.
    - If information is missing, write "Not provided".

    Return exactly:

    Order ID: <value>
    Product: <value>
    Issue: <value>
    Customer Request: <value>
    Date: <value>
    """

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return response.text

In [13]:
processed_result = process_complaint(complaint)

print("Processed Information:")
print(processed_result)

Processed Information:
Order ID: Not provided
Product: Not provided
Issue: Amount not refunded
Customer Request: Refund
Date: Not provided


In [14]:
resolution_prompt = f"""
You are an e-commerce customer support resolution system.

Determine the appropriate resolution for the customer's complaint.

Customer Complaint:
{complaint}

Complaint Classification:
{classification_result}

Extracted Information:
{processed_result}

Based on all the information above:

1. Identify the customer's main issue.
2. Determine the appropriate resolution.
3. Explain briefly why this resolution is appropriate.

Do not invent order details, policies, refunds, prices, or guarantees.

Return exactly:

Issue: <main issue>
Resolution: <recommended resolution>
Reason: <brief reason>
"""

In [15]:
resolution_response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=resolution_prompt
)

print("Resolution:")
print(resolution_response.text)

Resolution:
Issue: The customer has not received their refund.
Resolution: Request the necessary order details (such as Order ID and transaction information) from the customer to locate their account and investigate the refund status.
Reason: Essential information like the Order ID is missing, making it impossible to locate the purchase or verify the refund status without additional details from the customer.


In [18]:
def complaint_resolution_chain(complaint):

    # Step 1: Classification
    classification = classify_complaint(complaint)

    # Step 2: Information Extraction
    processed = process_complaint(complaint)

    # Step 3: Resolution
    resolution_prompt = f"""
    You are an e-commerce customer support resolution system.

    Customer Complaint:
    {complaint}

    Complaint Classification:
    {classification}

    Extracted Information:
    {processed}

    Determine the appropriate resolution.

    Rules:
    - Use only the information provided.
    - Do not invent order details.
    - Do not invent company policies.
    - Do not guarantee refunds or replacements.
    - If information is insufficient, clearly state that additional
      information or policy verification is required.

    Return exactly:

    Issue: <main issue>
    Resolution: <recommended resolution>
    Reason: <brief reason>
    """

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=resolution_prompt
    )

    return classification, processed, response.text

In [21]:
complaint = input("Enter your complaint: ")

classification, processed, resolution = complaint_resolution_chain(complaint)

print("\n CLASSIFICATION:")
print(classification)

print("\n PROCESSED INFORMATION:")
print(processed)

print("\n RESOLUTION:")
print(resolution)

Enter your complaint:  I ordered a black Nike shoe last week the order id is #12345, but I received the wrong size. I want to exchange it.



 CLASSIFICATION:
Category: Return/Replacement
Priority: Medium

 PROCESSED INFORMATION:
Order ID: #12345
Product: black Nike shoe
Issue: received the wrong size
Customer Request: exchange it
Date: last week

 RESOLUTION:
Issue: received the wrong size for order #12345 (black Nike shoe) and requested an exchange
Resolution: Process the exchange request for the correct size pending policy verification.
Reason: The customer received an incorrect size and requested an exchange, but policy verification is required to confirm eligibility and process the replacement.
